# 09-02. Two-stage Integration
## Model A × Model B로 실제 preseason 전체 선수 예측

---

지금까지는 두 모델을 따로 검증했습니다.

```text
Model A
P(다음 시즌 Big5 기록 존재)

Model B
E(다음 시즌 득점 | Big5 기록 존재)
```

이번 Notebook에서 처음으로 둘을 결합합니다.

---

# 핵심 질문

실제 preseason에는:

> "이 선수가 다음 시즌에도 Big5에서 뛸지"

를 확정적으로 알 수 없습니다.

따라서 현재 Big5 선수 전체를 대상으로:

```text
Soft Gate
P(Big5 presence)
×
Conditional Goal Prediction
```

또는:

```text
Hard Gate

P(Big5 presence) < threshold
→ 0골

P(Big5 presence) >= threshold
→ Model B prediction
```

을 사용합니다.

---

# 이번 실험의 Target

프로젝트 정의에 따라:

```text
next_goals_audited
```

를 전체 population의 실제 target으로 사용합니다.

즉:

```text
다음 시즌 Big5 record 없음
→ 다음 시즌 Big5 득점 = 0
```

입니다.

---

# 비교 방법

이번에는 단순히 Two-stage끼리만 비교하지 않습니다.

```text
1. Current_Goals_Naive
2. Conditional_No_Gate
3. TwoStage_Soft
4. TwoStage_Hard
5. Direct_Regression
```

### Current_Goals_Naive

현재 시즌 골을 그대로 다음 시즌 예측으로 사용.

### Conditional_No_Gate

Model B를 전체 선수에게 그대로 적용.

> Big5 이탈 가능성을 전혀 반영하지 않았을 때의 기준.

### TwoStage_Soft

```text
P(Big5) × conditional goals
```

### TwoStage_Hard

```text
P(Big5) >= threshold
→ conditional goals
else
→ 0
```

### Direct Regression

전체 population에 대해 `next_goals_audited`를 직접 회귀.

> Two-stage가 정말 필요한지 확인하기 위한 강한 baseline.

---

# 중요

Soft Gate는 전체 MAE를 개선할 수 있지만,
고득점 선수 예측까지 확률만큼 축소할 수 있습니다.

따라서:

- 전체 MAE
- Big5 Exit
- Big5 Presence
- 10+/15+/20+
- changed-team

을 반드시 같이 봅니다.

Final Test는 계속 LOCKED입니다.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 0. VS Code Colab / Google Drive

이 Notebook은 프로젝트 폴더명에 `(1)`이 붙어도 자동으로 찾습니다.

예:

```text
next_season_goal_prediction
next_season_goal_prediction (1)
next_season_goal_prediction (2)
```

중 `09_01A_snapshot_conservative_labels.csv`가 실제로 존재하는
`artifacts` 폴더를 자동 선택합니다.

In [4]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        "Google Drive가 마운트되어 있지 않습니다.\n"
        "VS Code에서 Ctrl + Shift + P → "
        "'Colab: Mount Google Drive to Server...'를 실행하세요."
    )

print(
    "Google Drive mounted:",
    DRIVE_ROOT.exists(),
)

Google Drive mounted: True


In [5]:
import json
import random
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    r2_score,
    roc_auc_score,
)

warnings.filterwarnings(
    "ignore"
)

SEED = 42

random.seed(
    SEED
)

np.random.seed(
    SEED
)

pd.set_option(
    "display.max_columns",
    120,
)

pd.set_option(
    "display.max_rows",
    200,
)

## 1. CatBoost / GPU

In [6]:
try:
    import catboost

    from catboost import (
        CatBoostClassifier,
        CatBoostRegressor,
    )

except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "catboost",
        ]
    )

    import catboost

    from catboost import (
        CatBoostClassifier,
        CatBoostRegressor,
    )


try:
    import torch

    USE_GPU = bool(
        torch.cuda.is_available()
    )

except ImportError:
    USE_GPU = False


CATBOOST_DEVICE_PARAMS = (
    {
        "task_type": "GPU",
        "devices": "0",
    }
    if USE_GPU
    else {
        "task_type": "CPU",
    }
)

print(
    "CatBoost:",
    catboost.__version__,
)

print(
    "GPU:",
    USE_GPU,
)

CatBoost: 1.2.10
GPU: True


# Part A. Artifact 자동 탐색

In [7]:
REQUIRED_SNAPSHOT = (
    "09_01A_snapshot_conservative_labels.csv"
)


def find_artifact_dir():
    candidates = []

    for project_dir in (
        DRIVE_ROOT.glob(
            "next_season_goal_prediction*"
        )
    ):
        artifact_dir = (
            project_dir
            / "artifacts"
        )

        snapshot_path = (
            artifact_dir
            / REQUIRED_SNAPSHOT
        )

        if snapshot_path.exists():
            candidates.append(
                artifact_dir
            )

    if not candidates:
        raise FileNotFoundError(
            f"{REQUIRED_SNAPSHOT}를 포함한 "
            "프로젝트 artifacts 폴더를 찾지 못했습니다."
        )

    print(
        "Found artifact dirs:"
    )

    for p in (
        candidates
    ):
        print(
            " -",
            p,
        )

    # 후보가 여러 개면 수정 시간이 가장 최근인 snapshot 선택
    candidates.sort(
        key=lambda p:
            (
                p
                / REQUIRED_SNAPSHOT
            ).stat().st_mtime,
        reverse=True,
    )

    return candidates[0]


ARTIFACT_DIR = (
    find_artifact_dir()
)

SNAPSHOT_PATH = (
    ARTIFACT_DIR
    / REQUIRED_SNAPSHOT
)

print(
    "\nSelected:",
    ARTIFACT_DIR,
)

print(
    "Snapshot:",
    SNAPSHOT_PATH,
)

print(
    "Exists:",
    SNAPSHOT_PATH.exists(),
)

Found artifact dirs:
 - /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts

Selected: /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts
Snapshot: /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_snapshot_conservative_labels.csv
Exists: True


In [8]:
df = pd.read_csv(
    SNAPSHOT_PATH,
    low_memory=False,
)

print(
    "Shape:",
    df.shape,
)

print(
    "Audited corrections:",
    int(
        df[
            "label_audit_changed"
        ].sum()
    ),
)

Shape: (23353, 124)
Audited corrections: 20


## 2. Final Test Lock

In [9]:
LOCKED_TEST_INPUT_SEASON = (
    "2024-2025"
)

LOCKED_TEST_TARGET_SEASON = (
    "2025-2026"
)

assert (
    LOCKED_TEST_INPUT_SEASON
    not in set(
        df[
            "season"
        ].astype(str)
    )
)

print(
    "✅ Final Test locked:",
    LOCKED_TEST_INPUT_SEASON,
    "→",
    LOCKED_TEST_TARGET_SEASON,
)

✅ Final Test locked: 2024-2025 → 2025-2026


# Part B. Historical Features

In [10]:
df[
    "season_start"
] = (
    df[
        "season"
    ]
    .astype(str)
    .str[:4]
    .astype(int)
)

df[
    "target_year"
] = (
    df[
        "target_season"
    ]
    .astype(str)
    .str[:4]
    .astype(int)
)

df = (
    df
    .sort_values(
        [
            "player",
            "season_start",
        ]
    )
    .reset_index(
        drop=True
    )
)

player_group = (
    df.groupby(
        "player",
        sort=False,
    )
)

df[
    "goals_3yr_mean"
] = player_group[
    "goals"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            3,
            min_periods=1,
        )
        .mean()
)

df[
    "goals_per90_3yr_mean"
] = player_group[
    "goals_per90"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            3,
            min_periods=1,
        )
        .mean()
)

df[
    "goals_3yr_max"
] = player_group[
    "goals"
].transform(
    lambda s:
        s.shift(1)
        .rolling(
            3,
            min_periods=1,
        )
        .max()
)

HISTORICAL_COLS = [
    "goals_3yr_mean",
    "goals_per90_3yr_mean",
    "goals_3yr_max",
]

df[
    HISTORICAL_COLS
] = (
    df[
        HISTORICAL_COLS
    ]
    .fillna(0.0)
)

# Part C. Feature Sets

In [11]:
BASE_NUMERIC = [
    "age",
    "starts",
    "minutes",
    "goals",
    "assists",
    "non_penalty_goals",
    "penalty_goals",
    "penalty_attempts",
    "goals_per90",
    "assists_per90",
    "goal_contrib_per90",
]

TEAM_BASE_FEATURES = [
    "old_team_rank_pct",
    "old_team_points_per_game",
    "old_team_goal_diff_per_game",
]

MARKET_FEATURES = [
    "market_value_known",
    "log_market_value",
    "market_value_percentile",
    "market_value_vs_position_median",
]

MOMENTUM_FEATURES = [
    "market_value_growth_6m",
    "market_value_growth_12m",
    "market_value_vs_peak",
]

TRANSFER_STATE_FEATURES = [
    "changed_team_preseason",
    "same_league_transfer",
    "league_changed",
    "country_changed",
    "is_loan_preseason",
    "days_since_transfer",
]

NEW_TEAM_FEATURES = [
    "new_team_prev_rank_pct",
    "new_team_prev_points_per_game",
    "new_team_prev_goal_diff_per_game",
    "team_rank_change",
    "team_points_change",
    "team_goal_diff_change",
    "new_team_strength_missing",
]

CATEGORICAL = [
    "league",
    "position_group",
]

S3_NUMERIC = (
    BASE_NUMERIC
    + HISTORICAL_COLS
    + TEAM_BASE_FEATURES
    + MARKET_FEATURES
    + MOMENTUM_FEATURES
    + TRANSFER_STATE_FEATURES
    + NEW_TEAM_FEATURES
)

MODEL_A_SPECIFIC = [
    "transfer_event_preseason",
    "destination_in_big5",
]

MODEL_A_NUMERIC = (
    S3_NUMERIC
    + MODEL_A_SPECIFIC
)

# Direct Regression은 Two-stage와 공정하게 비교하기 위해
# Model A가 사용하는 preseason-known 두 피처도 함께 허용
DIRECT_NUMERIC = (
    S3_NUMERIC
    + MODEL_A_SPECIFIC
)

print(
    "Model A numeric:",
    len(
        MODEL_A_NUMERIC
    ),
)

print(
    "Model B numeric:",
    len(
        S3_NUMERIC
    ),
)

print(
    "Direct numeric:",
    len(
        DIRECT_NUMERIC
    ),
)

Model A numeric: 39
Model B numeric: 37
Direct numeric: 39


## 3. Leakage Guard

In [12]:
FORBIDDEN = {
    "matched_next",
    "matched_next_original",
    "matched_next_audited",
    "next_goals",
    "next_goals_original",
    "next_goals_audited",
    "next_10plus",
    "next_10plus_original",
    "next_10plus_audited",
    "label_audit_changed",
    "label_audit_rule",
}

for feature_set in [
    MODEL_A_NUMERIC,
    S3_NUMERIC,
    DIRECT_NUMERIC,
]:
    assert not (
        FORBIDDEN
        & set(
            feature_set
        )
    )

print(
    "✅ No label/future target in features."
)

✅ No label/future target in features.


# Part D. Development Population

In [13]:
dev = (
    df[
        df[
            "target_year"
        ].ge(2017)
    ]
    .copy()
)

dev[
    "target_presence"
] = (
    dev[
        "matched_next_audited"
    ]
    .astype(int)
)

dev[
    "target_goals"
] = (
    dev[
        "next_goals_audited"
    ]
    .astype(float)
)

print(
    "Rows:",
    len(
        dev
    ),
)

print(
    "Presence rate:",
    f"{dev['target_presence'].mean():.2%}"
)

print(
    "Zero-target rate:",
    f"{dev['target_goals'].eq(0).mean():.2%}"
)

Rows: 7572
Presence rate: 84.40%
Zero-target rate: 36.77%


# Part E. Temporal Protocol

In [14]:
OUTER_VAL_SEASONS = [
    "2020-2021",
    "2021-2022",
    "2022-2023",
    "2023-2024",
]


def split_outer(
    data,
    val_season,
):
    val_start = int(
        val_season[:4]
    )

    train_df = (
        data[
            data[
                "season_start"
            ]
            < val_start
        ]
        .copy()
    )

    val_df = (
        data[
            data[
                "season_start"
            ]
            == val_start
        ]
        .copy()
    )

    return (
        train_df,
        val_df,
    )


def split_calibration(
    outer_train_df,
):
    cal_val_start = (
        outer_train_df[
            "season_start"
        ].max()
    )

    cal_train = (
        outer_train_df[
            outer_train_df[
                "season_start"
            ]
            < cal_val_start
        ]
        .copy()
    )

    cal_val = (
        outer_train_df[
            outer_train_df[
                "season_start"
            ]
            == cal_val_start
        ]
        .copy()
    )

    if (
        cal_train.empty
        or cal_val.empty
    ):
        raise ValueError(
            "Calibration split failure"
        )

    return (
        cal_train,
        cal_val,
    )

# Part F. CatBoost Preparation

In [15]:
def prepare_X(
    data,
    numeric_features,
):
    X = (
        data[
            numeric_features
            + CATEGORICAL
        ]
        .copy()
    )

    for col in (
        CATEGORICAL
    ):
        X[col] = (
            X[col]
            .fillna(
                "__MISSING__"
            )
            .astype(str)
        )

    for col in (
        numeric_features
    ):
        X[col] = pd.to_numeric(
            X[col],
            errors="coerce",
        )

    cat_indices = [
        X.columns.get_loc(
            col
        )
        for col in (
            CATEGORICAL
        )
    ]

    return (
        X,
        cat_indices,
    )

# Part G. Model A

In [16]:
A_MAX_ITER = 2000
A_PATIENCE = 60

A_PARAMS = {
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
}

THRESHOLD_GRID = np.round(
    np.arange(
        0.20,
        0.951,
        0.01,
    ),
    2,
)


def threshold_metrics(
    y_true,
    prob,
    threshold,
):
    y = np.asarray(
        y_true,
        dtype=int,
    )

    pred = (
        np.asarray(
            prob
        )
        >= threshold
    ).astype(int)

    (
        precision,
        recall,
        f1,
        support,
    ) = (
        precision_recall_fscore_support(
            y,
            pred,
            labels=[
                0,
                1,
            ],
            zero_division=0,
        )
    )

    return {
        "balanced_accuracy": (
            balanced_accuracy_score(
                y,
                pred,
            )
        ),
        "macro_f1": (
            f1_score(
                y,
                pred,
                average="macro",
                zero_division=0,
            )
        ),
        "exit_recall": (
            recall[0]
        ),
        "presence_recall": (
            recall[1]
        ),
    }


def choose_threshold(
    y_true,
    prob,
):
    rows = []

    for th in (
        THRESHOLD_GRID
    ):
        metrics = (
            threshold_metrics(
                y_true,
                prob,
                th,
            )
        )

        rows.append({
            "threshold": th,
            **metrics,
        })

    table = pd.DataFrame(
        rows
    )

    best = (
        table
        .sort_values(
            [
                "macro_f1",
                "balanced_accuracy",
                "presence_recall",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .iloc[0]
    )

    return float(
        best[
            "threshold"
        ]
    )

In [17]:
def fit_model_a(
    outer_train,
    outer_val,
    seed,
):
    (
        cal_train,
        cal_val,
    ) = split_calibration(
        outer_train
    )

    (
        X_cal_train,
        cat_indices,
    ) = prepare_X(
        cal_train,
        MODEL_A_NUMERIC,
    )

    (
        X_cal_val,
        _,
    ) = prepare_X(
        cal_val,
        MODEL_A_NUMERIC,
    )

    selector = (
        CatBoostClassifier(
            iterations=A_MAX_ITER,
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **CATBOOST_DEVICE_PARAMS,
            **A_PARAMS,
        )
    )

    selector.fit(
        X_cal_train,
        cal_train[
            "target_presence"
        ],
        cat_features=(
            cat_indices
        ),
        eval_set=(
            X_cal_val,
            cal_val[
                "target_presence"
            ],
        ),
        early_stopping_rounds=(
            A_PATIENCE
        ),
        use_best_model=True,
        verbose=False,
    )

    iterations = max(
        1,
        int(
            selector.get_best_iteration()
        )
        + 1,
    )

    cal_prob = (
        selector.predict_proba(
            X_cal_val
        )[:, 1]
    )

    threshold = (
        choose_threshold(
            cal_val[
                "target_presence"
            ],
            cal_prob,
        )
    )

    (
        X_train,
        cat_indices,
    ) = prepare_X(
        outer_train,
        MODEL_A_NUMERIC,
    )

    (
        X_val,
        _,
    ) = prepare_X(
        outer_val,
        MODEL_A_NUMERIC,
    )

    model = (
        CatBoostClassifier(
            iterations=iterations,
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **CATBOOST_DEVICE_PARAMS,
            **A_PARAMS,
        )
    )

    model.fit(
        X_train,
        outer_train[
            "target_presence"
        ],
        cat_features=(
            cat_indices
        ),
        verbose=False,
    )

    prob = (
        model.predict_proba(
            X_val
        )[:, 1]
    )

    return {
        "prob": prob,
        "threshold": threshold,
        "iterations": iterations,
    }

# Part H. Conditional Model B

In [18]:
B_MAX_ITER = 2000
B_PATIENCE = 50

B_PARAMS = {
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "RMSE",
}


def fit_conditional_model_b(
    outer_train_all,
    outer_val_all,
    seed,
):
    # Model B는 audited matched_next=True만 학습
    positive_train = (
        outer_train_all[
            outer_train_all[
                "target_presence"
            ].eq(1)
        ]
        .copy()
    )

    (
        cal_train,
        cal_val,
    ) = split_calibration(
        positive_train
    )

    (
        X_cal_train,
        cat_indices,
    ) = prepare_X(
        cal_train,
        S3_NUMERIC,
    )

    (
        X_cal_val,
        _,
    ) = prepare_X(
        cal_val,
        S3_NUMERIC,
    )

    selector = (
        CatBoostRegressor(
            iterations=B_MAX_ITER,
            eval_metric="MAE",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **CATBOOST_DEVICE_PARAMS,
            **B_PARAMS,
        )
    )

    selector.fit(
        X_cal_train,
        cal_train[
            "target_goals"
        ],
        cat_features=(
            cat_indices
        ),
        eval_set=(
            X_cal_val,
            cal_val[
                "target_goals"
            ],
        ),
        early_stopping_rounds=(
            B_PATIENCE
        ),
        use_best_model=True,
        verbose=False,
    )

    iterations = max(
        1,
        int(
            selector.get_best_iteration()
        )
        + 1,
    )

    (
        X_train,
        cat_indices,
    ) = prepare_X(
        positive_train,
        S3_NUMERIC,
    )

    # 중요:
    # 학습은 positive population만 하지만
    # prediction은 Outer Val 전체 선수에게 생성
    (
        X_val_all,
        _,
    ) = prepare_X(
        outer_val_all,
        S3_NUMERIC,
    )

    model = (
        CatBoostRegressor(
            iterations=iterations,
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **CATBOOST_DEVICE_PARAMS,
            **B_PARAMS,
        )
    )

    model.fit(
        X_train,
        positive_train[
            "target_goals"
        ],
        cat_features=(
            cat_indices
        ),
        verbose=False,
    )

    pred = np.clip(
        model.predict(
            X_val_all
        ),
        0,
        None,
    )

    return {
        "pred": pred,
        "iterations": iterations,
        "train_positive_n": len(
            positive_train
        ),
    }

# Part I. Direct Regression Baseline

In [19]:
D_MAX_ITER = 2000
D_PATIENCE = 50

D_PARAMS = {
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "RMSE",
}


def fit_direct_regression(
    outer_train,
    outer_val,
    seed,
):
    (
        cal_train,
        cal_val,
    ) = split_calibration(
        outer_train
    )

    (
        X_cal_train,
        cat_indices,
    ) = prepare_X(
        cal_train,
        DIRECT_NUMERIC,
    )

    (
        X_cal_val,
        _,
    ) = prepare_X(
        cal_val,
        DIRECT_NUMERIC,
    )

    selector = (
        CatBoostRegressor(
            iterations=D_MAX_ITER,
            eval_metric="MAE",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **CATBOOST_DEVICE_PARAMS,
            **D_PARAMS,
        )
    )

    selector.fit(
        X_cal_train,
        cal_train[
            "target_goals"
        ],
        cat_features=(
            cat_indices
        ),
        eval_set=(
            X_cal_val,
            cal_val[
                "target_goals"
            ],
        ),
        early_stopping_rounds=(
            D_PATIENCE
        ),
        use_best_model=True,
        verbose=False,
    )

    iterations = max(
        1,
        int(
            selector.get_best_iteration()
        )
        + 1,
    )

    (
        X_train,
        cat_indices,
    ) = prepare_X(
        outer_train,
        DIRECT_NUMERIC,
    )

    (
        X_val,
        _,
    ) = prepare_X(
        outer_val,
        DIRECT_NUMERIC,
    )

    model = (
        CatBoostRegressor(
            iterations=iterations,
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **CATBOOST_DEVICE_PARAMS,
            **D_PARAMS,
        )
    )

    model.fit(
        X_train,
        outer_train[
            "target_goals"
        ],
        cat_features=(
            cat_indices
        ),
        verbose=False,
    )

    pred = np.clip(
        model.predict(
            X_val
        ),
        0,
        None,
    )

    return {
        "pred": pred,
        "iterations": iterations,
    }

# Part J. Metrics

In [20]:
def evaluate_regression(
    y_true,
    pred,
    presence,
):
    y = np.asarray(
        y_true,
        dtype=float,
    )

    p = np.asarray(
        pred,
        dtype=float,
    )

    presence = np.asarray(
        presence,
        dtype=int,
    )

    out = {
        "n": len(
            y
        ),
        "mae": (
            mean_absolute_error(
                y,
                p,
            )
        ),
        "rmse": (
            mean_squared_error(
                y,
                p,
            )
            ** 0.5
        ),
        "r2": (
            r2_score(
                y,
                p,
            )
        ),
        "bias": float(
            np.mean(
                p - y
            )
        ),
    }

    exit_mask = (
        presence
        == 0
    )

    retain_mask = (
        presence
        == 1
    )

    out[
        "exit_n"
    ] = int(
        exit_mask.sum()
    )

    out[
        "exit_mae"
    ] = (
        mean_absolute_error(
            y[
                exit_mask
            ],
            p[
                exit_mask
            ],
        )
        if exit_mask.any()
        else np.nan
    )

    out[
        "exit_pred_mean"
    ] = (
        float(
            p[
                exit_mask
            ].mean()
        )
        if exit_mask.any()
        else np.nan
    )

    out[
        "presence_n"
    ] = int(
        retain_mask.sum()
    )

    out[
        "presence_mae"
    ] = (
        mean_absolute_error(
            y[
                retain_mask
            ],
            p[
                retain_mask
            ],
        )
        if retain_mask.any()
        else np.nan
    )

    for threshold in [
        10,
        15,
        20,
    ]:
        mask = (
            y
            >= threshold
        )

        out[
            f"{threshold}plus_n"
        ] = int(
            mask.sum()
        )

        out[
            f"{threshold}plus_mae"
        ] = (
            mean_absolute_error(
                y[
                    mask
                ],
                p[
                    mask
                ],
            )
            if mask.any()
            else np.nan
        )

        out[
            f"{threshold}plus_bias"
        ] = (
            float(
                np.mean(
                    p[
                        mask
                    ]
                    - y[
                        mask
                    ]
                )
            )
            if mask.any()
            else np.nan
        )

    return out

In [21]:
def evaluate_classifier(
    y_true,
    prob,
    threshold,
):
    y = np.asarray(
        y_true,
        dtype=int,
    )

    p = np.clip(
        np.asarray(
            prob,
            dtype=float,
        ),
        1e-7,
        1 - 1e-7,
    )

    t = (
        threshold_metrics(
            y,
            p,
            threshold,
        )
    )

    return {
        "log_loss": (
            log_loss(
                y,
                p,
                labels=[
                    0,
                    1,
                ],
            )
        ),
        "brier": (
            brier_score_loss(
                y,
                p,
            )
        ),
        "roc_auc": (
            roc_auc_score(
                y,
                p,
            )
        ),
        "pr_auc_exit": (
            average_precision_score(
                1 - y,
                1 - p,
            )
        ),
        **t,
    }

# Part K. Walk-forward 실행

In [22]:
result_rows = []
classifier_rows = []
prediction_frames = []
fold_config_rows = []


for fold, val_season in enumerate(
    OUTER_VAL_SEASONS,
    start=1,
):
    (
        outer_train,
        outer_val,
    ) = split_outer(
        dev,
        val_season,
    )

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"Fold {fold} | "
        f"Val {val_season} | "
        f"Train {len(outer_train):,} | "
        f"Val {len(outer_val):,}"
    )

    # -------------------------
    # Model A
    # -------------------------
    a_result = (
        fit_model_a(
            outer_train,
            outer_val,
            seed=(
                SEED
                + fold * 100
            ),
        )
    )

    # -------------------------
    # Conditional Model B
    # -------------------------
    b_result = (
        fit_conditional_model_b(
            outer_train,
            outer_val,
            seed=(
                SEED
                + fold * 1000
            ),
        )
    )

    # -------------------------
    # Direct Regression
    # -------------------------
    d_result = (
        fit_direct_regression(
            outer_train,
            outer_val,
            seed=(
                SEED
                + fold * 10000
            ),
        )
    )

    y = (
        outer_val[
            "target_goals"
        ]
        .to_numpy(
            dtype=float
        )
    )

    presence = (
        outer_val[
            "target_presence"
        ]
        .to_numpy(
            dtype=int
        )
    )

    current_goals = (
        outer_val[
            "goals"
        ]
        .to_numpy(
            dtype=float
        )
    )

    prob = (
        a_result[
            "prob"
        ]
    )

    conditional = (
        b_result[
            "pred"
        ]
    )

    threshold = (
        a_result[
            "threshold"
        ]
    )

    soft = (
        prob
        * conditional
    )

    hard = np.where(
        prob
        >= threshold,
        conditional,
        0.0,
    )

    direct = (
        d_result[
            "pred"
        ]
    )

    methods = {
        "Current_Goals_Naive": (
            current_goals
        ),
        "Conditional_No_Gate": (
            conditional
        ),
        "TwoStage_Soft": (
            soft
        ),
        "TwoStage_Hard": (
            hard
        ),
        "Direct_Regression": (
            direct
        ),
    }

    for (
        method,
        pred,
    ) in (
        methods.items()
    ):
        metrics = (
            evaluate_regression(
                y,
                pred,
                presence,
            )
        )

        result_rows.append({
            "method": method,
            "fold": fold,
            "outer_val_season": (
                val_season
            ),
            **metrics,
        })

        print(
            f"{method:<22} | "
            f"MAE {metrics['mae']:.4f} | "
            f"Exit {metrics['exit_mae']:.4f} | "
            f"Presence {metrics['presence_mae']:.4f} | "
            f"10+ {metrics['10plus_mae']:.4f}"
        )

    class_metrics = (
        evaluate_classifier(
            presence,
            prob,
            threshold,
        )
    )

    classifier_rows.append({
        "fold": fold,
        "outer_val_season": (
            val_season
        ),
        "threshold": threshold,
        **class_metrics,
    })

    fold_config_rows.append({
        "fold": fold,
        "outer_val_season": (
            val_season
        ),
        "model_a_iterations": (
            a_result[
                "iterations"
            ]
        ),
        "model_a_threshold": (
            threshold
        ),
        "model_b_iterations": (
            b_result[
                "iterations"
            ]
        ),
        "model_b_positive_train_n": (
            b_result[
                "train_positive_n"
            ]
        ),
        "direct_iterations": (
            d_result[
                "iterations"
            ]
        ),
    })

    pred_frame = (
        outer_val[
            [
                "row_id",
                "player",
                "team",
                "league",
                "season",
                "target_season",
                "position_group",
                "goals",
                "target_presence",
                "target_goals",
                "changed_team_preseason",
                "destination_in_big5",
                "label_audit_changed",
            ]
        ]
        .copy()
    )

    pred_frame[
        "presence_probability"
    ] = prob

    pred_frame[
        "presence_threshold"
    ] = threshold

    pred_frame[
        "conditional_goal_pred"
    ] = conditional

    pred_frame[
        "two_stage_soft_pred"
    ] = soft

    pred_frame[
        "two_stage_hard_pred"
    ] = hard

    pred_frame[
        "direct_regression_pred"
    ] = direct

    prediction_frames.append(
        pred_frame
    )


outer_results = (
    pd.DataFrame(
        result_rows
    )
)

classifier_results = (
    pd.DataFrame(
        classifier_rows
    )
)

fold_config = (
    pd.DataFrame(
        fold_config_rows
    )
)

predictions = (
    pd.concat(
        prediction_frames,
        ignore_index=True,
    )
)


Fold 1 | Val 2020-2021 | Train 3,817 | Val 960


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


Current_Goals_Naive    | MAE 2.5354 | Exit 1.9127 | Presence 2.6295 | 10+ 5.0326
Conditional_No_Gate    | MAE 2.2717 | Exit 1.8450 | Presence 2.3362 | 10+ 5.8904
TwoStage_Soft          | MAE 2.1055 | Exit 0.9686 | Presence 2.2773 | 10+ 6.0978
TwoStage_Hard          | MAE 2.0751 | Exit 0.6419 | Presence 2.2916 | 10+ 5.9309
Direct_Regression      | MAE 2.1319 | Exit 1.0730 | Presence 2.2919 | 10+ 6.0696

Fold 2 | Val 2021-2022 | Train 4,777 | Val 934


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


Current_Goals_Naive    | MAE 2.8480 | Exit 2.3724 | Presence 2.9354 | 10+ 6.2911
Conditional_No_Gate    | MAE 2.4261 | Exit 2.1150 | Presence 2.4833 | 10+ 6.6487
TwoStage_Soft          | MAE 2.2537 | Exit 1.2307 | Presence 2.4417 | 10+ 6.7928
TwoStage_Hard          | MAE 2.2638 | Exit 1.0764 | Presence 2.4821 | 10+ 6.6487
Direct_Regression      | MAE 2.2769 | Exit 1.3740 | Presence 2.4429 | 10+ 6.6968

Fold 3 | Val 2022-2023 | Train 5,711 | Val 938


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


Current_Goals_Naive    | MAE 2.7878 | Exit 2.9529 | Presence 2.7456 | 10+ 5.5256
Conditional_No_Gate    | MAE 2.3819 | Exit 2.6443 | Presence 2.3148 | 10+ 6.4708
TwoStage_Soft          | MAE 2.1260 | Exit 1.4742 | Presence 2.2926 | 10+ 6.6256
TwoStage_Hard          | MAE 2.0596 | Exit 1.0942 | Presence 2.3065 | 10+ 6.4708
Direct_Regression      | MAE 2.1762 | Exit 1.7628 | Presence 2.2819 | 10+ 6.5130

Fold 4 | Val 2023-2024 | Train 6,649 | Val 923


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


Current_Goals_Naive    | MAE 2.8353 | Exit 2.9545 | Presence 2.8114 | 10+ 5.6437
Conditional_No_Gate    | MAE 2.4621 | Exit 2.4963 | Presence 2.4552 | 10+ 6.7181
TwoStage_Soft          | MAE 2.1851 | Exit 1.0420 | Presence 2.4140 | 10+ 6.9195
TwoStage_Hard          | MAE 2.1081 | Exit 0.4303 | Presence 2.4441 | 10+ 6.7997
Direct_Regression      | MAE 2.2225 | Exit 1.2616 | Presence 2.4149 | 10+ 6.9723


# Part L. Overall Summary

In [23]:
summary = (
    outer_results
    .groupby(
        "method"
    )
    .agg(
        folds=(
            "fold",
            "nunique",
        ),
        mae_mean=(
            "mae",
            "mean",
        ),
        mae_std=(
            "mae",
            "std",
        ),
        mae_worst=(
            "mae",
            "max",
        ),
        rmse_mean=(
            "rmse",
            "mean",
        ),
        r2_mean=(
            "r2",
            "mean",
        ),
        bias_mean=(
            "bias",
            "mean",
        ),
        exit_mae_mean=(
            "exit_mae",
            "mean",
        ),
        exit_pred_mean=(
            "exit_pred_mean",
            "mean",
        ),
        presence_mae_mean=(
            "presence_mae",
            "mean",
        ),
        tenplus_mae_mean=(
            "10plus_mae",
            "mean",
        ),
        fifteenplus_mae_mean=(
            "15plus_mae",
            "mean",
        ),
        twentyplus_mae_mean=(
            "20plus_mae",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        "mae_mean"
    )
)

summary

,method,folds,mae_mean,mae_std,mae_worst,rmse_mean,r2_mean,bias_mean,exit_mae_mean,exit_pred_mean,presence_mae_mean,tenplus_mae_mean,fifteenplus_mae_mean,twentyplus_mae_mean
3,TwoStage_Hard,4,2.126666,0.093655,2.263846,3.258930,0.474459,0.037049,0.810697,0.810697,2.381066,6.462509,8.627918,10.853575
4,TwoStage_Soft,4,2.167563,0.066601,2.253697,3.243437,0.479495,0.021081,1.178902,1.178902,2.356390,6.608930,8.789650,11.044500
2,Direct_Regression,4,2.201881,0.062218,2.276932,3.279600,0.467947,0.048125,1.367862,1.367862,2.357890,6.562908,8.825399,11.118459
0,Conditional_No_Gate,4,2.385456,0.082597,2.462079,3.393504,0.430177,0.370073,2.275156,2.275156,2.397380,6.431992,8.627918,10.853575
1,Current_Goals_Naive,4,2.751637,0.146452,2.847966,4.037661,0.192255,1.015304,2.548134,2.548134,2.780488,5.623267,7.093246,8.277404


## Model A Summary

In [24]:
classifier_summary = pd.DataFrame({
    "metric": [
        "log_loss",
        "brier",
        "roc_auc",
        "pr_auc_exit",
        "balanced_accuracy",
        "macro_f1",
        "exit_recall",
        "presence_recall",
        "threshold",
    ],
    "mean": [
        classifier_results[
            "log_loss"
        ].mean(),
        classifier_results[
            "brier"
        ].mean(),
        classifier_results[
            "roc_auc"
        ].mean(),
        classifier_results[
            "pr_auc_exit"
        ].mean(),
        classifier_results[
            "balanced_accuracy"
        ].mean(),
        classifier_results[
            "macro_f1"
        ].mean(),
        classifier_results[
            "exit_recall"
        ].mean(),
        classifier_results[
            "presence_recall"
        ].mean(),
        classifier_results[
            "threshold"
        ].mean(),
    ],
})

classifier_summary

,metric,mean
0,log_loss,0.236096
1,brier,0.067393
2,roc_auc,0.923522
3,pr_auc_exit,0.776512
4,balanced_accuracy,0.830652
5,macro_f1,0.833073
6,exit_recall,0.709783
7,presence_recall,0.951521
8,threshold,0.652500


# Part M. Gate 효과 직접 비교

In [25]:
conditional_summary = (
    summary[
        summary[
            "method"
        ].eq(
            "Conditional_No_Gate"
        )
    ]
    .iloc[0]
)

gate_delta_rows = []

for method in [
    "TwoStage_Soft",
    "TwoStage_Hard",
]:
    row = (
        summary[
            summary[
                "method"
            ].eq(
                method
            )
        ]
        .iloc[0]
    )

    gate_delta_rows.append({
        "method": method,
        "mae_delta_vs_no_gate": (
            row[
                "mae_mean"
            ]
            - conditional_summary[
                "mae_mean"
            ]
        ),
        "exit_mae_delta_vs_no_gate": (
            row[
                "exit_mae_mean"
            ]
            - conditional_summary[
                "exit_mae_mean"
            ]
        ),
        "presence_mae_delta_vs_no_gate": (
            row[
                "presence_mae_mean"
            ]
            - conditional_summary[
                "presence_mae_mean"
            ]
        ),
        "10plus_mae_delta_vs_no_gate": (
            row[
                "tenplus_mae_mean"
            ]
            - conditional_summary[
                "tenplus_mae_mean"
            ]
        ),
        "20plus_mae_delta_vs_no_gate": (
            row[
                "twentyplus_mae_mean"
            ]
            - conditional_summary[
                "twentyplus_mae_mean"
            ]
        ),
    })


gate_delta = pd.DataFrame(
    gate_delta_rows
)

gate_delta

,method,mae_delta_vs_no_gate,exit_mae_delta_vs_no_gate,presence_mae_delta_vs_no_gate,10plus_mae_delta_vs_no_gate,20plus_mae_delta_vs_no_gate
0,TwoStage_Soft,-0.217893,-1.096254,-0.040990,0.176938,0.190924
1,TwoStage_Hard,-0.258790,-1.464459,-0.016314,0.030516,0.000000


# Part N. Pooled OOF Slice Analysis

In [26]:
def pooled_metric(
    actual,
    pred,
):
    actual = np.asarray(
        actual,
        dtype=float,
    )

    pred = np.asarray(
        pred,
        dtype=float,
    )

    return {
        "n": len(
            actual
        ),
        "mae": (
            mean_absolute_error(
                actual,
                pred,
            )
        ),
        "bias": float(
            np.mean(
                pred - actual
            )
        ),
        "actual_mean": float(
            actual.mean()
        ),
        "pred_mean": float(
            pred.mean()
        ),
    }


PRED_COLS = {
    "Current_Goals_Naive": (
        "goals"
    ),
    "Conditional_No_Gate": (
        "conditional_goal_pred"
    ),
    "TwoStage_Soft": (
        "two_stage_soft_pred"
    ),
    "TwoStage_Hard": (
        "two_stage_hard_pred"
    ),
    "Direct_Regression": (
        "direct_regression_pred"
    ),
}

## 1. Actual Big5 Exit Slice

In [27]:
exit_rows = (
    predictions[
        predictions[
            "target_presence"
        ].eq(0)
    ]
)

exit_slice_rows = []

for method, col in (
    PRED_COLS.items()
):
    m = pooled_metric(
        exit_rows[
            "target_goals"
        ],
        exit_rows[
            col
        ],
    )

    exit_slice_rows.append({
        "method": method,
        **m,
    })


exit_slice_summary = (
    pd.DataFrame(
        exit_slice_rows
    )
    .sort_values(
        "mae"
    )
)

exit_slice_summary

,method,n,mae,bias,actual_mean,pred_mean
3,TwoStage_Hard,616,0.831517,0.831517,0.0,0.831517
2,TwoStage_Soft,616,1.205442,1.205442,0.0,1.205442
4,Direct_Regression,616,1.404900,1.404900,0.0,1.404900
1,Conditional_No_Gate,616,2.319221,2.319221,0.0,2.319221
0,Current_Goals_Naive,616,2.603896,2.603896,0.0,2.603896


## 2. Actual Big5 Presence Slice

In [28]:
presence_rows = (
    predictions[
        predictions[
            "target_presence"
        ].eq(1)
    ]
)

presence_slice_rows = []

for method, col in (
    PRED_COLS.items()
):
    m = pooled_metric(
        presence_rows[
            "target_goals"
        ],
        presence_rows[
            col
        ],
    )

    presence_slice_rows.append({
        "method": method,
        **m,
    })


presence_slice_summary = (
    pd.DataFrame(
        presence_slice_rows
    )
    .sort_values(
        "mae"
    )
)

presence_slice_summary

,method,n,mae,bias,actual_mean,pred_mean
2,TwoStage_Soft,3139,2.355741,-0.211112,3.779548,3.568435
4,Direct_Regression,3139,2.357597,-0.217886,3.779548,3.561662
3,TwoStage_Hard,3139,2.380380,-0.117962,3.779548,3.661586
1,Conditional_No_Gate,3139,2.397239,-0.012628,3.779548,3.766919
0,Current_Goals_Naive,3139,2.778592,0.702772,3.779548,4.482319


## 3. High Scorer Slice

In [29]:
high_scorer_rows = []

for threshold in [
    10,
    15,
    20,
]:
    subset = (
        predictions[
            predictions[
                "target_goals"
            ].ge(
                threshold
            )
        ]
    )

    for method, col in (
        PRED_COLS.items()
    ):
        m = pooled_metric(
            subset[
                "target_goals"
            ],
            subset[
                col
            ],
        )

        high_scorer_rows.append({
            "threshold": threshold,
            "method": method,
            **m,
        })


high_scorer_summary = (
    pd.DataFrame(
        high_scorer_rows
    )
)

display(
    high_scorer_summary
)

,threshold,method,n,mae,bias,actual_mean,pred_mean
0,10,Current_Goals_Naive,336,5.601190,-3.357143,14.425595,11.068452
1,10,Conditional_No_Gate,336,6.417734,-5.970666,14.425595,8.454929
2,10,TwoStage_Soft,336,6.596498,-6.197107,14.425595,8.228488
3,10,TwoStage_Hard,336,6.449942,-6.002875,14.425595,8.422721
4,10,Direct_Regression,336,6.553716,-6.112909,14.425595,8.312686
5,15,Current_Goals_Naive,126,7.103175,-4.722222,19.452381,14.730159
6,15,Conditional_No_Gate,126,8.634890,-8.291640,19.452381,11.160741
7,15,TwoStage_Soft,126,8.796311,-8.511998,19.452381,10.940383
8,15,TwoStage_Hard,126,8.634890,-8.291640,19.452381,11.160741
9,15,Direct_Regression,126,8.833344,-8.455768,19.452381,10.996613


## 4. Changed-team Slice

In [30]:
changed_rows = (
    predictions[
        predictions[
            "changed_team_preseason"
        ].eq(1)
    ]
)

changed_team_rows = []

for method, col in (
    PRED_COLS.items()
):
    m = pooled_metric(
        changed_rows[
            "target_goals"
        ],
        changed_rows[
            col
        ],
    )

    changed_team_rows.append({
        "method": method,
        **m,
    })


changed_team_summary = (
    pd.DataFrame(
        changed_team_rows
    )
    .sort_values(
        "mae"
    )
)

changed_team_summary

,method,n,mae,bias,actual_mean,pred_mean
3,TwoStage_Hard,455,2.291497,-0.046772,3.07033,3.023557
2,TwoStage_Soft,455,2.424030,0.025947,3.07033,3.096277
4,Direct_Regression,455,2.591308,0.171640,3.07033,3.241970
1,Conditional_No_Gate,455,2.949189,0.721765,3.07033,3.792095
0,Current_Goals_Naive,455,3.457143,1.694505,3.07033,4.764835


# Part O. 대표 Error Cases

Soft Gate가 high scorer를 과도하게 줄이는지,
Hard Gate가 실제 presence 선수를 0으로 만드는지 확인합니다.

In [31]:
error_view = (
    predictions.copy()
)

error_view[
    "soft_abs_error"
] = (
    error_view[
        "two_stage_soft_pred"
    ]
    - error_view[
        "target_goals"
    ]
).abs()

error_view[
    "hard_abs_error"
] = (
    error_view[
        "two_stage_hard_pred"
    ]
    - error_view[
        "target_goals"
    ]
).abs()

display(
    error_view[
        error_view[
            "target_goals"
        ].ge(10)
    ][
        [
            "player",
            "season",
            "target_season",
            "team",
            "target_goals",
            "presence_probability",
            "presence_threshold",
            "conditional_goal_pred",
            "two_stage_soft_pred",
            "two_stage_hard_pred",
            "direct_regression_pred",
            "soft_abs_error",
            "hard_abs_error",
        ]
    ]
    .sort_values(
        "target_goals",
        ascending=False,
    )
    .head(50)
)

,player,season,target_season,team,target_goals,presence_probability,presence_threshold,conditional_goal_pred,two_stage_soft_pred,two_stage_hard_pred,direct_regression_pred,soft_abs_error,hard_abs_error
1172,Erling Haaland,2021-2022,2022-2023,Dortmund,36.0,0.998217,0.59,17.917935,17.885988,17.917935,17.670689,18.114012,18.082065
2225,Harry Kane,2022-2023,2023-2024,Tottenham,36.0,0.972909,0.64,19.833451,19.296150,19.833451,18.078308,16.703850,16.166549
755,Robert Lewandowski,2020-2021,2021-2022,Bayern Munich,35.0,0.979833,0.62,28.318347,27.747255,28.318347,28.546590,7.252745,6.681653
3307,Kylian Mbappé,2023-2024,2024-2025,Paris S-G,31.0,0.998628,0.76,25.884016,25.848516,25.884016,25.150969,5.151484,5.115984
1264,Harry Kane,2021-2022,2022-2023,Tottenham,30.0,0.984415,0.59,15.745109,15.499723,15.745109,15.287128,14.500277,14.254891
1431,Kylian Mbappé,2021-2022,2022-2023,Paris S-G,29.0,0.996699,0.59,24.149414,24.069698,24.149414,24.582715,4.930302,4.850586
3453,Mohamed Salah,2023-2024,2024-2025,Liverpool,29.0,0.971028,0.76,17.162216,16.664994,17.162216,16.988667,12.335006,11.837784
475,Kylian Mbappé,2020-2021,2021-2022,Paris S-G,28.0,0.993050,0.62,24.353728,24.184482,24.353728,23.428056,3.815518,3.646272
2707,Serhou Guirassy,2022-2023,2023-2024,Stuttgart,28.0,0.988388,0.64,7.231299,7.147329,7.231299,7.120015,20.852671,20.768701
3577,Robert Lewandowski,2023-2024,2024-2025,Barcelona,27.0,0.957397,0.76,15.646676,14.980077,15.646676,15.236053,12.019923,11.353324


## 실제 Presence인데 Hard Gate로 0이 된 선수

In [32]:
false_exit_gate = (
    predictions[
        predictions[
            "target_presence"
        ].eq(1)
        & (
            predictions[
                "presence_probability"
            ]
            < predictions[
                "presence_threshold"
            ]
        )
    ]
    .copy()
)

display(
    false_exit_gate[
        [
            "player",
            "season",
            "target_season",
            "team",
            "target_goals",
            "presence_probability",
            "presence_threshold",
            "conditional_goal_pred",
            "two_stage_hard_pred",
        ]
    ]
    .sort_values(
        "target_goals",
        ascending=False,
    )
    .head(50)
)

,player,season,target_season,team,target_goals,presence_probability,presence_threshold,conditional_goal_pred,two_stage_hard_pred
287,Gianluca Caprari,2020-2021,2021-2022,Benevento,12.0,0.483161,0.62,3.726287,0.0
2889,Andrea Pinamonti,2023-2024,2024-2025,Sassuolo,10.0,0.683603,0.76,7.095901,0.0
2954,Boulaye Dia,2023-2024,2024-2025,Salernitana,9.0,0.559713,0.76,4.887453,0.0
2509,Milan Đurić,2022-2023,2023-2024,Hellas Verona,9.0,0.519758,0.64,1.304706,0.0
2842,Adrien Rabiot,2023-2024,2024-2025,Juventus,9.0,0.477127,0.76,2.827309,0.0
823,Sergi Guardiola,2020-2021,2021-2022,Valladolid,8.0,0.425180,0.62,1.792808,0.0
1118,David Okereke,2021-2022,2022-2023,Venezia,7.0,0.421874,0.59,3.677951,0.0
2278,James Ward-Prowse,2022-2023,2023-2024,Southampton,7.0,0.432389,0.64,5.346119,0.0
1739,Saîf-Eddine Khaoui,2021-2022,2022-2023,Clermont Foot,7.0,0.539667,0.59,1.609647,0.0
2417,Lucas Boyé,2022-2023,2023-2024,Elche,6.0,0.600810,0.64,4.632249,0.0


# Part P. 결과 저장

In [33]:
OUTPUTS = {
    "outer_results": (
        ARTIFACT_DIR
        / "09_02_two_stage_outer_results.csv"
    ),
    "summary": (
        ARTIFACT_DIR
        / "09_02_two_stage_summary.csv"
    ),
    "predictions": (
        ARTIFACT_DIR
        / "09_02_two_stage_predictions.csv"
    ),
    "classifier_results": (
        ARTIFACT_DIR
        / "09_02_model_a_outer_results.csv"
    ),
    "classifier_summary": (
        ARTIFACT_DIR
        / "09_02_model_a_summary.csv"
    ),
    "fold_config": (
        ARTIFACT_DIR
        / "09_02_fold_config.csv"
    ),
    "gate_delta": (
        ARTIFACT_DIR
        / "09_02_gate_delta.csv"
    ),
    "exit_slice": (
        ARTIFACT_DIR
        / "09_02_exit_slice_summary.csv"
    ),
    "presence_slice": (
        ARTIFACT_DIR
        / "09_02_presence_slice_summary.csv"
    ),
    "high_scorer": (
        ARTIFACT_DIR
        / "09_02_high_scorer_summary.csv"
    ),
    "changed_team": (
        ARTIFACT_DIR
        / "09_02_changed_team_summary.csv"
    ),
}


outer_results.to_csv(
    OUTPUTS[
        "outer_results"
    ],
    index=False,
)

summary.to_csv(
    OUTPUTS[
        "summary"
    ],
    index=False,
)

predictions.to_csv(
    OUTPUTS[
        "predictions"
    ],
    index=False,
)

classifier_results.to_csv(
    OUTPUTS[
        "classifier_results"
    ],
    index=False,
)

classifier_summary.to_csv(
    OUTPUTS[
        "classifier_summary"
    ],
    index=False,
)

fold_config.to_csv(
    OUTPUTS[
        "fold_config"
    ],
    index=False,
)

gate_delta.to_csv(
    OUTPUTS[
        "gate_delta"
    ],
    index=False,
)

exit_slice_summary.to_csv(
    OUTPUTS[
        "exit_slice"
    ],
    index=False,
)

presence_slice_summary.to_csv(
    OUTPUTS[
        "presence_slice"
    ],
    index=False,
)

high_scorer_summary.to_csv(
    OUTPUTS[
        "high_scorer"
    ],
    index=False,
)

changed_team_summary.to_csv(
    OUTPUTS[
        "changed_team"
    ],
    index=False,
)


print(
    "Saved:"
)

for name, path in (
    OUTPUTS.items()
):
    print(
        f"- {name:<20}",
        path.resolve(),
    )

Saved:
- outer_results        /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_02_two_stage_outer_results.csv
- summary              /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_02_two_stage_summary.csv
- predictions          /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_02_two_stage_predictions.csv
- classifier_results   /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_02_model_a_outer_results.csv
- classifier_summary   /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_02_model_a_summary.csv
- fold_config          /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_02_fold_config.csv
- gate_delta           /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_02_gate_delta.csv
- exit_slice           /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_02_exit_slice_summary.csv
- presence_slice       /content/drive/MyDrive/next_season_goal_pr

## Protocol

In [34]:
PROTOCOL = {
    "stage": (
        "09-02 Two-stage Integration"
    ),

    "source_snapshot": (
        SNAPSHOT_PATH.name
    ),

    "label_policy": (
        "09-01A conservative audited labels"
    ),

    "population": (
        "all development rows target_year >= 2017"
    ),

    "target": (
        "next_goals_audited"
    ),

    "model_a": {
        "type": (
            "Fixed CatBoost Classifier"
        ),
        "target": (
            "matched_next_audited"
        ),
        "features": (
            "S3 + transfer_event_preseason "
            "+ destination_in_big5"
        ),
        "threshold_selection": (
            "last season inside Outer Train; "
            "maximize Macro F1"
        ),
    },

    "model_b": {
        "type": (
            "Fixed CatBoost Regressor S3"
        ),
        "training_population": (
            "matched_next_audited == True"
        ),
        "target": (
            "next_goals_audited"
        ),
        "prediction_population": (
            "all Outer Validation rows"
        ),
    },

    "two_stage_methods": {
        "soft": (
            "presence_probability * conditional_goal_pred"
        ),
        "hard": (
            "conditional_goal_pred if "
            "presence_probability >= inner-selected threshold "
            "else 0"
        ),
    },

    "direct_baseline": {
        "type": (
            "Fixed CatBoost Regressor"
        ),
        "population": (
            "all rows"
        ),
        "target": (
            "next_goals_audited"
        ),
        "features": (
            "S3 + transfer_event_preseason "
            "+ destination_in_big5"
        ),
    },

    "outer_validation_seasons": (
        OUTER_VAL_SEASONS
    ),

    "test_input_season": (
        LOCKED_TEST_INPUT_SEASON
    ),

    "test_target_season": (
        LOCKED_TEST_TARGET_SEASON
    ),

    "test_status": (
        "LOCKED / NOT LOADED"
    ),
}

PROTOCOL_PATH = (
    ARTIFACT_DIR
    / "09_02_protocol.json"
)

with PROTOCOL_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        PROTOCOL,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    PROTOCOL_PATH.resolve()
)

/content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_02_protocol.json


# 결과 해석 체크리스트

실행 후 아래 질문에 답합니다.

---

## A. Two-stage 자체가 효과가 있는가?

`Conditional_No_Gate` 대비:

- Soft Gate 전체 MAE는 좋아졌는가?
- Hard Gate 전체 MAE는 좋아졌는가?
- Exit MAE는 얼마나 좋아졌는가?
- Presence MAE는 얼마나 나빠졌는가?

---

## B. Soft vs Hard

### Soft가 좋다면

확률을 그대로 기대값에 반영하는 것이 유효.

### Hard가 좋다면

Big5 Presence를 확률 가중치보다
**선별 gate**로 사용하는 편이 유효.

---

## C. Direct Regression

가장 중요한 비교입니다.

```text
Two-stage
vs
Direct Regression
```

Direct가 더 좋다면:

> 복잡한 Two-stage 구조가 반드시 필요한 것은 아님.

Two-stage가 더 좋다면:

> Big5 존재 여부와 조건부 득점을 분리한 구조가 실제로 의미 있음.

---

## D. High Scorer

Soft Gate가 10+/20+ 예측을 더 낮춰서
기존 underprediction을 악화시키는지 확인.

이 경우:

```text
전체 MAE용 모델
≠
고득점자 탐지/예측용 모델
```

이라는 결론이 더 강해집니다.

---

# 실행 후 보내줄 파일

```text
09_02_two_stage_summary.csv
09_02_gate_delta.csv
09_02_exit_slice_summary.csv
09_02_presence_slice_summary.csv
09_02_high_scorer_summary.csv
09_02_two_stage_predictions.csv
```

이 결과로 09 단계의 최종 구조를 결정합니다.